## Phase VI Work Report: MEV Classification and Trader Profiling

The primary objective of this phase was to identify Maximum Extractable Value actors and victims, and subsequently aggregate millions of isolated decentralized exchange swaps into comprehensive, individual trader profiles. I designed this architecture to analyze trader sophistication, market diversification, and execution quality across the Ethereum network.

## Methodology

1. **MEV Detection and Classification:**  
   I built a deterministic detection algorithm utilizing Dataset E. By tracking directional token flows within the same block and pool, I isolated sandwich attacks where a bot brackets a victim transaction. Furthermore, I identified cyclic arbitrage by tracing multi-leg transactions where the starting asset perfectly matched the ending asset. I output these classifications into a standalone MEV dataset.

2. **Trader Aggregation:**  
   Utilizing the canonical swap ledger, grouped all transactions by unique wallet addresses. I engineered dozens of behavioral metrics for each trader, calculating their total volume, average trade size, operational tenure, and active days.

3. **Diversification and Concentration:**  
   To measure trader sophistication, I implemented the Herfindahl-Hirschman Index for both capital allocation and router utilization. This allowed me to mathematically quantify whether a trader was highly concentrated in a single liquidity pool or heavily diversified across the ecosystem.

4. **Execution Quality and Typology:**  
   merged the precise price impact calculations and transaction gas costs into the trader profiles. Finally, I authored a heuristic classification engine to segment the population into distinct typologies, categorizing them as MEV bots, whales, high-frequency traders, active retail, or casual retail.


In [1]:
from config import OUT
import polars as pl
from pathlib import Path
import gc
import os

print(f"Saving data to: {OUT}")

Saving data to: C:\Users\Pouyan\python\thesis\Proposal\FINAL\Thesis_Output


Goal: Classify MEV (Dataset F) and begin building Trader Profiles (Dataset G).

---
## Mev Classification and Dataset F Construction
This cell identifies Maximum Extractable Value activity by analyzing intra block token flows. It uses a lazy query plan to protect system memory. By grouping trades within the same block and pool, it detects sandwich attacks by observing opposing directional trades wrapping a central victim. It also identifies cyclic arbitrage and prioritizes the classifications before exporting Dataset F.

In [7]:
OUT_DIR = Path("./Thesis_Output")
E_PATH = OUT_DIR / "Dataset_E_PoolState.parquet"

print("Step 1: Setting up Lazy Query Plan...")

# utilize lazy scanning to prevent memory exhaustion when processing massive files
lf = pl.scan_parquet(E_PATH).select(
    ["transaction_hash", "block_number", "log_index", "pool_address", "tx_from", "amount0_f64"]
)

print("Step 2: Building Mev detection logic...")

# assign numeric directions based on capital flow
lf = lf.with_columns(
    pl.when(pl.col("amount0_f64") > 0).then(pl.lit(1)).otherwise(pl.lit(-1)).alias("dir")
)

# isolate sandwich bots by finding direction reversals within the same block
bot_activity = lf.group_by(["block_number", "pool_address", "tx_from"]).agg([
    pl.col("log_index").min().alias("front_log"),
    pl.col("log_index").max().alias("back_log"),
    pl.col("dir").first().alias("front_dir"),
    pl.col("dir").last().alias("back_dir"),
]).filter(
    (pl.col("front_log") < pl.col("back_log")) & 
    (pl.col("front_dir") != pl.col("back_dir"))
).rename({"tx_from": "bot_address"})

# attach the identified bots back to the primary trade ledger
joined = lf.join(bot_activity, on=["block_number", "pool_address"], how="inner")

# classify victims trapped between the front run and back run logs
victims = joined.filter(
    (pl.col("log_index") > pl.col("front_log")) & 
    (pl.col("log_index") < pl.col("back_log")) & 
    (pl.col("dir") == pl.col("front_dir")) & 
    (pl.col("tx_from") != pl.col("bot_address"))
).with_columns(pl.lit("sandwich_victim").alias("mev_type"))

# classify the attackers initiating the bracket trades
attackers = joined.filter(
    (pl.col("tx_from") == pl.col("bot_address")) & 
    ((pl.col("log_index") == pl.col("front_log")) | (pl.col("log_index") == pl.col("back_log")))
).with_columns(pl.lit("sandwich_attacker").alias("mev_type"))

# identify arbitrageurs trading across multiple pools simultaneously
arb_activity = lf.group_by(["block_number", "tx_from"]).agg([
    pl.col("pool_address").n_unique().alias("n_pools")
]).filter(pl.col("n_pools") > 1)

sandwich_bot_addrs = bot_activity.select("bot_address").unique().rename({"bot_address": "tx_from"})
pure_arbs = arb_activity.join(sandwich_bot_addrs, on="tx_from", how="anti")

arbitrage_trades = lf.join(pure_arbs, on=["block_number", "tx_from"], how="inner") \
    .with_columns(pl.lit("arbitrage").alias("mev_type"))

# merge all classifications into a single unified frame
classified = pl.concat([
    victims.select(["transaction_hash", "block_number", "log_index", "mev_type", pl.col("bot_address").alias("extractor_address")]),
    attackers.select(["transaction_hash", "block_number", "log_index", "mev_type", pl.col("bot_address").alias("extractor_address")]),
    arbitrage_trades.select(["transaction_hash", "block_number", "log_index", "mev_type", pl.col("tx_from").alias("extractor_address")])
])

# prioritize classifications to handle edge cases where roles overlap
classified = classified.with_columns(
    pl.col("mev_type").replace(
        {"sandwich_victim": 1, "sandwich_attacker": 2, "arbitrage": 3}, 
        return_dtype=pl.Int32
    ).alias("priority")
).sort(["transaction_hash", "log_index", "priority"]).unique(subset=["transaction_hash", "log_index"], keep="first")

# establish the final dataset schema with boolean flags for downstream modeling
F_lazy = classified.with_columns([
    pl.lit(None, dtype=pl.Float64).alias("estimated_loss_usd"), 
    (pl.col("mev_type") == "sandwich_victim").alias("is_sandwich_victim"),
    (pl.col("mev_type") == "sandwich_attacker").alias("is_sandwich_attacker"),
    (pl.col("mev_type") == "arbitrage").alias("is_arbitrage"),
    pl.lit(False).alias("is_liquidation"), 
    pl.lit(False).alias("is_backrun"),     
]).select([
    "transaction_hash", "block_number", "log_index", "mev_type",
    "estimated_loss_usd", "extractor_address",
    "is_sandwich_victim", "is_sandwich_attacker",
    "is_arbitrage", "is_liquidation", "is_backrun"
])

print(" Executing Query Plan... (This may take several minutes, but will protect your RAM)")

# collect triggers the lazy execution plan
F = F_lazy.collect() 

print(" Processing complete! Saving files...")

parquet_path = OUT_DIR / "Dataset_F_MEV.parquet"
F.write_parquet(parquet_path, compression="zstd")
print(f" Saved Parquet to: {parquet_path}")

gc.collect() 

csv_path = OUT_DIR / "Dataset_F_MEV.csv"
F.write_csv(csv_path)
print(f" Saved Csv to: {csv_path}")

print("\n--- Mev Classification Results ---")
print(F.group_by("mev_type").len().sort("len", descending=True))

Step 1: Setting up Lazy Query Plan...
Step 2: Building Mev detection logic...
 Executing Query Plan... (This may take several minutes, but will protect your RAM)


C:\Users\Pouyan\AppData\Local\Temp\ipykernel_23572\4284098359.py:66: DeprecationWarning: the `return_dtype` parameter for `replace` is deprecated. Use `replace_strict` instead to set a return data type while replacing values.
(Deprecated in version 1.0.0)
  pl.col("mev_type").replace(


 Processing complete! Saving files...
 Saved Parquet to: Thesis_Output\Dataset_F_MEV.parquet
 Saved Csv to: Thesis_Output\Dataset_F_MEV.csv

--- Mev Classification Results ---
shape: (3, 2)
┌───────────────────┬──────────┐
│ mev_type          ┆ len      │
│ ---               ┆ ---      │
│ str               ┆ u32      │
╞═══════════════════╪══════════╡
│ arbitrage         ┆ 12274231 │
│ sandwich_attacker ┆ 1149116  │
│ sandwich_victim   ┆ 423683   │
└───────────────────┴──────────┘


---
## Trader History Configuration

This cell defines the global configurations and file paths necessary to construct the trader history panel. It establishes quality control thresholds, such as voiding impossible fiat valuations, and provides a version tolerant collection helper function to manage large data streams safely.

In [28]:
OUT = Path("./Thesis_Output")

A_PATH     = OUT / "Dataset_A_Final.parquet"
POOLS_PATH = OUT / "pools_v1.parquet"
F_PATH     = OUT / "Dataset_F_MEV.parquet"
G_PATH     = OUT / "Dataset_G_TraderHistory.parquet"

# strict quality gates to sanitize the dataset before aggregation
DROP_MALFORMED   = True        
NULL_ON_OVERFLOW = True        
MAX_USD_SANITY   = 1e10        
PROJECTS         = None        

# define the primary key for trader identification
TRADER_KEY = "wallet"          

def collect(lf, label=""):
    # custom streaming collector to accommodate different polars version syntaxes
    if label:
        print(f"  {label} ...")
    try:
        return lf.collect(engine="streaming")
    except TypeError:
        return lf.collect(streaming=True)

print("Config loaded.")

Config loaded.


---
## Pool Registry Ingestion
This cell loads the previously validated pool registry and selects only the essential columns required for merging. This slim profile drastically reduces memory overhead during the massive join operations coming in subsequent cells.

In [17]:
# standardize casing for reliable table joining
pools = pl.read_parquet(POOLS_PATH).with_columns([
    pl.col("pool_address").str.to_lowercase(),
    pl.col("token0").str.to_lowercase(),
    pl.col("token1").str.to_lowercase(),
])

# select a restricted schema to optimize memory consumption using the correct fee_pct column
pools_slim = pools.select([
    "pool_address",
    pl.col("token0").alias("t0"),
    pl.col("token1").alias("t1"),
    pl.col("decimals0").alias("dec0"),
    pl.col("decimals1").alias("dec1"),
    "fee_tier", pl.col("fee_pct").alias("fee_frac"), "symbol0", "symbol1", "price_safe",
])

print(f" {pools_slim.height:,} pools loaded")
print(pools_slim.head(5))
print("fee tier distribution")
print(pools.group_by("fee_tier").len().sort("fee_tier"))
print(f"unsafe price pools {pools.filter(~pl.col('price_safe')).height:,}")

 66,327 pools loaded
shape: (5, 10)
┌─────────────┬─────────────┬─────────────┬──────┬───┬──────────┬───────────┬─────────┬────────────┐
│ pool_addres ┆ t0          ┆ t1          ┆ dec0 ┆ … ┆ fee_frac ┆ symbol0   ┆ symbol1 ┆ price_safe │
│ s           ┆ ---         ┆ ---         ┆ ---  ┆   ┆ ---      ┆ ---       ┆ ---     ┆ ---        │
│ ---         ┆ str         ┆ str         ┆ i64  ┆   ┆ f64      ┆ str       ┆ str     ┆ bool       │
│ str         ┆             ┆             ┆      ┆   ┆          ┆           ┆         ┆            │
╞═════════════╪═════════════╪═════════════╪══════╪═══╪══════════╪═══════════╪═════════╪════════════╡
│ 0xa2e6466f1 ┆ 0x27916a293 ┆ 0xc02aaa39b ┆ 18   ┆ … ┆ 0.01     ┆ DEBATES   ┆ WETH    ┆ true       │
│ ff9d6965062 ┆ 2b8e6a80bdc ┆ 223fe8d0a0e ┆      ┆   ┆          ┆           ┆         ┆            │
│ 2ba8bd13…   ┆ ac7f67b2…   ┆ 5c4f27ea…   ┆      ┆   ┆          ┆           ┆         ┆            │
│ 0xfeb20fc60 ┆ 0xc02aaa39b ┆ 0xec6d73557 ┆ 18   ┆ … ┆ 

---
## Base Lazy Frame Initialization

This cell initializes the master view of all valid swaps. It filters out malformed transactions, enforces the fiat valuation sanity limits, and joins the slim pool registry to append fee tier data. Using lazy evaluation ensures that no computational resources are spent until the final aggregation commands are called.

In [20]:
usd_raw = pl.col("amount_usd").cast(pl.Float64).abs()

# enforce strict sanity limits to drop corrupted valuation artifacts
vol_usd = (
    pl.when(
        usd_raw.is_not_null()
        & usd_raw.is_finite()
        & (usd_raw > 0)
        & (usd_raw <= MAX_USD_SANITY)
        & (~pl.col("amount_overflow") if NULL_ON_OVERFLOW else pl.lit(True))
    )
    .then(usd_raw)
    .otherwise(None)
    .alias("vol_usd")
)

base = pl.scan_parquet(A_PATH)

# execute quality control filters
if DROP_MALFORMED:
    base = base.filter(~pl.col("malformed_legs").fill_null(False))
if PROJECTS is not None:
    base = base.filter(pl.col("project").is_in(PROJECTS))

# construct the base view joining essential pool attributes
base = (
    base
    .with_columns(pl.col("pool_address").str.to_lowercase())
    .join(pools_slim.lazy().select(["pool_address", "fee_tier"]),
          on="pool_address", how="left")
    .select([
        pl.col(TRADER_KEY).str.to_lowercase().alias("trader_address"),
        "pool_address", "tx_hash", "log_index",
        pl.col("block_number").cast(pl.Int64),
        pl.col("block_time").alias("ts"),
        pl.col("token_in").str.to_lowercase().alias("token_in"),
        pl.col("token_out").str.to_lowercase().alias("token_out"),
        "fee_tier",
        vol_usd,
    ])
    .filter(pl.col("trader_address").is_not_null())
)

print("Base view defined (lazy — nothing executed yet).")

Base view defined (lazy — nothing executed yet).


---
## Trader Aggregation and Behavioral Metrics
This cell compresses the transactional ledger into individual trader profiles. By grouping the data by wallet address, it calculates sophisticated behavioral metrics including trading tenure, operational scale, average trade sizes, and ecosystem diversification. Logarithmic transformations are applied to volume and count metrics to normalize the severe skew typical of financial data.

In [30]:
G = base.group_by("trader_address").agg([
    # capture temporal activity boundaries
    pl.col("ts").min().alias("first_trade_ts"),
    pl.col("ts").max().alias("last_trade_ts"),
    pl.col("block_number").min().alias("first_block"),
    pl.col("block_number").max().alias("last_block"),
    pl.len().alias("n_swaps"),
    pl.col("tx_hash").n_unique().alias("n_transactions"),
    pl.col("ts").dt.date().n_unique().alias("active_days"),
    pl.col("block_number").n_unique().alias("n_active_blocks"),

    # compute aggregate financial scale
    pl.col("vol_usd").sum().alias("total_volume_usd"),
    pl.col("vol_usd").is_not_null().sum().alias("n_swaps_priced"),

    # define statistical properties of the order sizes
    pl.col("vol_usd").mean().alias("avg_trade_size_usd"),
    pl.col("vol_usd").median().alias("median_trade_size_usd"),
    pl.col("vol_usd").std().alias("std_trade_size_usd"),
    pl.col("vol_usd").min().alias("min_trade_size_usd"),
    pl.col("vol_usd").max().alias("max_trade_size_usd"),
    pl.col("vol_usd").quantile(0.90).alias("p90_trade_size_usd"),

    # measure ecosystem exploration and diversification
    pl.col("pool_address").n_unique().alias("n_pools"),
    pl.col("fee_tier").n_unique().alias("n_fee_tiers"),
    pl.col("token_in").n_unique().alias("n_tokens_in"),
    pl.col("token_out").n_unique().alias("n_tokens_out"),
])

# calculate ratio based heuristics and logarithmic normalizations
G = G.with_columns(
    ((pl.col("last_trade_ts") - pl.col("first_trade_ts"))
        .dt.total_seconds() / 86400.0).alias("tenure_days")
).with_columns([
    (pl.col("n_swaps") / pl.col("active_days").cast(pl.Float64)).alias("trades_per_active_day"),
    (pl.col("n_swaps") / pl.col("n_transactions").cast(pl.Float64)).alias("swaps_per_tx"),
    (pl.col("std_trade_size_usd") / pl.col("avg_trade_size_usd")).alias("size_cv"),
    (pl.col("active_days").cast(pl.Float64) / (pl.col("tenure_days") + 1.0)).alias("activity_ratio"),
    (pl.col("n_pools").cast(pl.Float64) / pl.col("n_swaps")).alias("pool_swap_ratio"),
    (pl.col("n_swaps_priced").cast(pl.Float64) / pl.col("n_swaps")).alias("usd_coverage"),
    (pl.col("n_pools") > 1).alias("is_multi_pool"),
    (pl.col("n_swaps") == 1).alias("is_one_shot_trader"),
    pl.col("first_trade_ts").dt.truncate("1mo").alias("cohort_month"),
    pl.col("n_swaps").log1p().alias("log_n_swaps"),
    pl.col("total_volume_usd").log1p().alias("log_total_volume"),
    pl.col("avg_trade_size_usd").log1p().alias("log_avg_size"),
    pl.col("n_pools").log1p().alias("log_n_pools"),
    (pl.col("tenure_days") + 1).log().alias("log_tenure"),
])

G_df = collect(G, "trader aggregation")
print(f" {G_df.height:,} unique traders")
print(G_df.select(["n_swaps", "total_volume_usd", "avg_trade_size_usd",
                   "median_trade_size_usd", "tenure_days", "usd_coverage"]).describe())

  trader aggregation ...
 3,987,674 unique traders
shape: (9, 7)
┌────────────┬─────────────┬──────────────┬──────────────┬─────────────┬─────────────┬─────────────┐
│ statistic  ┆ n_swaps     ┆ total_volume ┆ avg_trade_si ┆ median_trad ┆ tenure_days ┆ usd_coverag │
│ ---        ┆ ---         ┆ _usd         ┆ ze_usd       ┆ e_size_usd  ┆ ---         ┆ e           │
│ str        ┆ f64         ┆ ---          ┆ ---          ┆ ---         ┆ f64         ┆ ---         │
│            ┆             ┆ f64          ┆ f64          ┆ f64         ┆             ┆ f64         │
╞════════════╪═════════════╪══════════════╪══════════════╪═════════════╪═════════════╪═════════════╡
│ count      ┆ 3.987674e6  ┆ 3.987674e6   ┆ 3.986759e6   ┆ 3.986759e6  ┆ 3.987674e6  ┆ 3.987674e6  │
│ null_count ┆ 0.0         ┆ 0.0          ┆ 915.0        ┆ 915.0       ┆ 0.0         ┆ 0.0         │
│ mean       ┆ 14.924976   ┆ 96569.379686 ┆ 1811.364813  ┆ 1548.062003 ┆ 48.896887   ┆ 0.999513    │
│ std        ┆ 2015.158704

---
## Trader Aggregation and Behavioral Metrics Duplicate
This cell performs the exact same aggregation routine as the previous cell. It is retained to maintain structural fidelity with the original notebook inputs.

In [33]:
G = base.group_by("trader_address").agg([
    # capture temporal activity boundaries
    pl.col("ts").min().alias("first_trade_ts"),
    pl.col("ts").max().alias("last_trade_ts"),
    pl.col("block_number").min().alias("first_block"),
    pl.col("block_number").max().alias("last_block"),
    pl.len().alias("n_swaps"),
    pl.col("tx_hash").n_unique().alias("n_transactions"),
    pl.col("ts").dt.date().n_unique().alias("active_days"),
    pl.col("block_number").n_unique().alias("n_active_blocks"),

    # compute aggregate financial scale
    pl.col("vol_usd").sum().alias("total_volume_usd"),
    pl.col("vol_usd").is_not_null().sum().alias("n_swaps_priced"),

    # define statistical properties of the order sizes
    pl.col("vol_usd").mean().alias("avg_trade_size_usd"),
    pl.col("vol_usd").median().alias("median_trade_size_usd"),
    pl.col("vol_usd").std().alias("std_trade_size_usd"),
    pl.col("vol_usd").min().alias("min_trade_size_usd"),
    pl.col("vol_usd").max().alias("max_trade_size_usd"),
    pl.col("vol_usd").quantile(0.90).alias("p90_trade_size_usd"),

    # measure ecosystem exploration and diversification
    pl.col("pool_address").n_unique().alias("n_pools"),
    pl.col("fee_tier").n_unique().alias("n_fee_tiers"),
    pl.col("token_in").n_unique().alias("n_tokens_in"),
    pl.col("token_out").n_unique().alias("n_tokens_out"),
])

# calculate ratio based heuristics and logarithmic normalizations
G = G.with_columns(
    ((pl.col("last_trade_ts") - pl.col("first_trade_ts"))
        .dt.total_seconds() / 86400.0).alias("tenure_days")
).with_columns([
    (pl.col("n_swaps") / pl.col("active_days").cast(pl.Float64)).alias("trades_per_active_day"),
    (pl.col("n_swaps") / pl.col("n_transactions").cast(pl.Float64)).alias("swaps_per_tx"),
    (pl.col("std_trade_size_usd") / pl.col("avg_trade_size_usd")).alias("size_cv"),
    (pl.col("active_days").cast(pl.Float64) / (pl.col("tenure_days") + 1.0)).alias("activity_ratio"),
    (pl.col("n_pools").cast(pl.Float64) / pl.col("n_swaps")).alias("pool_swap_ratio"),
    (pl.col("n_swaps_priced").cast(pl.Float64) / pl.col("n_swaps")).alias("usd_coverage"),
    (pl.col("n_pools") > 1).alias("is_multi_pool"),
    (pl.col("n_swaps") == 1).alias("is_one_shot_trader"),
    pl.col("first_trade_ts").dt.truncate("1mo").alias("cohort_month"),
    pl.col("n_swaps").log1p().alias("log_n_swaps"),
    pl.col("total_volume_usd").log1p().alias("log_total_volume"),
    pl.col("avg_trade_size_usd").log1p().alias("log_avg_size"),
    pl.col("n_pools").log1p().alias("log_n_pools"),
    (pl.col("tenure_days") + 1).log().alias("log_tenure"),
])

G_df = collect(G, "trader aggregation")
print(f" {G_df.height:,} unique traders")
print(G_df.select(["n_swaps", "total_volume_usd", "avg_trade_size_usd",
                   "median_trade_size_usd", "tenure_days", "usd_coverage"]).describe())

  trader aggregation ...
 3,987,674 unique traders
shape: (9, 7)
┌────────────┬─────────────┬──────────────┬──────────────┬─────────────┬─────────────┬─────────────┐
│ statistic  ┆ n_swaps     ┆ total_volume ┆ avg_trade_si ┆ median_trad ┆ tenure_days ┆ usd_coverag │
│ ---        ┆ ---         ┆ _usd         ┆ ze_usd       ┆ e_size_usd  ┆ ---         ┆ e           │
│ str        ┆ f64         ┆ ---          ┆ ---          ┆ ---         ┆ f64         ┆ ---         │
│            ┆             ┆ f64          ┆ f64          ┆ f64         ┆             ┆ f64         │
╞════════════╪═════════════╪══════════════╪══════════════╪═════════════╪═════════════╪═════════════╡
│ count      ┆ 3.987674e6  ┆ 3.987674e6   ┆ 3.986759e6   ┆ 3.986759e6  ┆ 3.987674e6  ┆ 3.987674e6  │
│ null_count ┆ 0.0         ┆ 0.0          ┆ 915.0        ┆ 915.0       ┆ 0.0         ┆ 0.0         │
│ mean       ┆ 14.924976   ┆ 96569.379686 ┆ 1811.364813  ┆ 1548.062003 ┆ 48.896887   ┆ 0.999513    │
│ std        ┆ 2015.158704

---
## Herfindahl Hirschman Index Calculation
This cell calculates the Herfindahl Hirschman Index for each trader. By squaring the market share of each liquidity pool utilized by a wallet, it creates a robust metric of diversification. A high index indicates a trader concentrated entirely in one or two pools, while a low index reveals a highly diversified market participant.

In [36]:
hhi = (
    base.with_columns(pl.col("vol_usd").fill_null(0.0))
        .group_by(["trader_address", "pool_address"])
        .agg([pl.col("vol_usd").sum().alias("pv"), pl.len().alias("pc")])
        .with_columns([
            # calculate the percentage share of each pool for the specific trader
            (pl.col("pv") / pl.col("pv").sum().over("trader_address")).alias("vshare"),
            (pl.col("pc") / pl.col("pc").sum().over("trader_address")).alias("cshare"),
        ])
        .group_by("trader_address")
        .agg([
            # square the shares to derive the concentration index
            (pl.col("vshare") ** 2).sum().alias("hhi_volume"),
            (pl.col("cshare") ** 2).sum().alias("hhi_count"),
            pl.col("vshare").max().alias("top_pool_vol_share"),
        ])
)

G_df = G_df.join(collect(hhi, "HHI"), on="trader_address", how="left")
gc.collect()
print(G_df.select(["hhi_volume", "hhi_count", "top_pool_vol_share"]).describe())

  HHI ...
shape: (9, 4)
┌────────────┬────────────┬────────────┬────────────────────┐
│ statistic  ┆ hhi_volume ┆ hhi_count  ┆ top_pool_vol_share │
│ ---        ┆ ---        ┆ ---        ┆ ---                │
│ str        ┆ f64        ┆ f64        ┆ f64                │
╞════════════╪════════════╪════════════╪════════════════════╡
│ count      ┆ 3.987674e6 ┆ 3.987674e6 ┆ 3.987674e6         │
│ null_count ┆ 0.0        ┆ 0.0        ┆ 0.0                │
│ mean       ┆ NaN        ┆ 0.729614   ┆ NaN                │
│ std        ┆ NaN        ┆ 0.316628   ┆ NaN                │
│ min        ┆ 0.000823   ┆ 0.00059    ┆ 0.005971           │
│ 25%        ┆ 0.500039   ┆ 0.5        ┆ 0.578325           │
│ 50%        ┆ 1.0        ┆ 1.0        ┆ 1.0                │
│ 75%        ┆ 1.0        ┆ 1.0        ┆ 1.0                │
│ max        ┆ 1.0        ┆ 1.0        ┆ 1.0                │
└────────────┴────────────┴────────────┴────────────────────┘


---
## Cyclic Arbitrage Detection

This cell detects true cyclic arbitrage by examining the start and end tokens of multileg transactions. If a transaction routes through multiple pools but ultimately outputs the exact same asset it started with, it is flagged as cyclic. This is a critical heuristic for identifying sophisticated bot behavior.

In [39]:
tx_paths = (
    base.sort(["tx_hash", "log_index"])
        .group_by("tx_hash")
        .agg([
            pl.col("trader_address").first(),
            # capture the entry and exit assets of the transaction
            pl.col("token_in").first().alias("path_start"),
            pl.col("token_out").last().alias("path_end"),
            pl.len().alias("legs"),
        ])
        .with_columns(
            # flag transactions that close the loop as cyclic arbitrage
            ((pl.col("path_start") == pl.col("path_end")) & (pl.col("legs") >= 2))
            .alias("is_cyclic")
        )
)
tx_paths_df = collect(tx_paths, "cyclic path detection")

n_tx = tx_paths_df.height
n_cyclic = int(tx_paths_df["is_cyclic"].sum())
print(f"total transactions {n_tx:,}")
print(f"cyclic arbitrage trades {n_cyclic:,} representing {100*n_cyclic/n_tx:.2f} percent")
print(f"multi leg trades {int((tx_paths_df['legs']>=2).sum()):,}")

# aggregate the cyclic behavior to the individual trader profile
cyc_by_trader = (
    tx_paths_df.group_by("trader_address")
    .agg([
        pl.col("is_cyclic").sum().alias("n_cyclic_tx"),
        pl.col("legs").max().alias("max_legs_in_tx"),
    ])
)

G_df = G_df.join(cyc_by_trader, on="trader_address", how="left").with_columns([
    pl.col("n_cyclic_tx").fill_null(0),
    pl.col("max_legs_in_tx").fill_null(1),
])

del tx_paths_df, cyc_by_trader
gc.collect()

  cyclic path detection ...
total transactions 45,532,663
cyclic arbitrage trades 1,009,597 representing 2.22 percent
multi leg trades 7,964,641


0

---
## Mev Coverage and Profiling Integration
This cell merges the Mev classifications from Dataset F into the primary trader profiles. Before executing the final aggregation, it runs a strict coverage diagnostic to verify that the transaction hashes match perfectly across datasets. It then calculates the proportional rate at which each trader engages in or falls victim to algorithmic extraction.

In [42]:
F = pl.scan_parquet(F_PATH).select([
    pl.col("transaction_hash").str.to_lowercase().alias("tx_hash"),
    "log_index", "mev_type", "estimated_loss_usd",
    "is_sandwich_victim", "is_sandwich_attacker", "is_arbitrage",
])

A_keys = base.select(["trader_address", "tx_hash", "log_index"]).with_columns(
    pl.col("tx_hash").str.to_lowercase()
)

# coverage diagnostic to prevent silent key mismatches
cov = collect(
    F.join(A_keys, on=["tx_hash", "log_index"], how="left")
     .select([
         pl.len().alias("f_rows"),
         pl.col("trader_address").is_null().sum().alias("unmatched"),
     ]),
    "mev join coverage"
)

print(cov)
u, t = cov["unmatched"][0], cov["f_rows"][0]
print(f"failed matches {100*u/t:.2f} percent" if u/t > 0.05 else f"join healthy {100*(1-u/t):.2f} percent matched")

# aggregate the isolated mev events to the trader level
mev_by_trader = (
    A_keys.join(F, on=["tx_hash", "log_index"], how="inner")
    .group_by("trader_address").agg([
        pl.col("is_sandwich_victim").fill_null(False).sum().alias("n_victim_trades"),
        pl.col("is_sandwich_attacker").fill_null(False).sum().alias("n_attacker_trades"),
        pl.col("is_arbitrage").fill_null(False).sum().alias("n_arbitrage_trades"),
        pl.col("estimated_loss_usd").sum().alias("mev_loss_usd"),
    ])
)

G_df = (
    G_df.join(collect(mev_by_trader, "mev by trader"), on="trader_address", how="left")
    .with_columns([
        pl.col("n_victim_trades").fill_null(0),
        pl.col("n_attacker_trades").fill_null(0),
        pl.col("n_arbitrage_trades").fill_null(0),
        pl.col("mev_loss_usd").fill_null(0.0),
    ])
    .with_columns([
        # calculate proportional exposure metrics for the final analysis
        (pl.col("n_victim_trades") / pl.col("n_swaps")).alias("victim_rate"),
        (pl.col("n_arbitrage_trades") / pl.col("n_swaps")).alias("arb_share"),
        (pl.col("n_cyclic_tx") / pl.col("n_transactions")).alias("cyclic_share"),
        (pl.col("mev_loss_usd") / pl.col("total_volume_usd")).alias("mev_loss_per_dollar"),
    ])
)
gc.collect()

  mev join coverage ...
shape: (1, 2)
┌──────────┬───────────┐
│ f_rows   ┆ unmatched │
│ ---      ┆ ---       │
│ u32      ┆ u32       │
╞══════════╪═══════════╡
│ 13847030 ┆ 1105164   │
└──────────┴───────────┘
failed matches 7.98 percent
  mev by trader ...


0

---
## Heuristic Trader Typology Classification
This cell applies an econometric heuristic to categorize the trader population into distinct typologies. It identifies algorithmic bots based on hyperactive cyclic execution and sandwich attack frequency. It subsequently partitions the remaining human participants into whales, high frequency traders, active retail, or casual retail based on their trading volume and operational frequency percentiles.

In [44]:
# flag automated algorithmic entities based on execution speed and mev behavior
G_df = G_df.with_columns(
    (
        (pl.col("n_attacker_trades") >= 2)                                  
        | ((pl.col("cyclic_share") > 0.50) & (pl.col("n_transactions") >= 20))
        | ((pl.col("swaps_per_tx") >= 2.5) & (pl.col("n_swaps") >= 50))
        | ((pl.col("trades_per_active_day") >= 100) & (pl.col("n_swaps") >= 500))
    ).alias("is_mev_bot")
)

print(f"mev bots identified {int(G_df['is_mev_bot'].sum()):,} representing {100*G_df['is_mev_bot'].mean():.4f} percent")

# define quantitative thresholds utilizing the ninety ninth percentile of activity
q = G_df.select([
    pl.col("avg_trade_size_usd").quantile(0.99).alias("size_p99"),
    pl.col("n_swaps").quantile(0.99).alias("swaps_p99"),
]).row(0)

print(f"classification cutoffs whale size exceeds {q[0]:,.0f} and high frequency exceeds {q[1]:,.0f} swaps")

# categorize the population into strict econometric typologies
G_df = G_df.with_columns(
    pl.when(pl.col("is_mev_bot")).then(pl.lit("mev_bot"))
     .when(pl.col("n_swaps") == 1).then(pl.lit("one_shot"))
     .when(pl.col("n_swaps") >= q[1]).then(pl.lit("high_frequency"))
     .when(pl.col("avg_trade_size_usd") >= q[0]).then(pl.lit("whale"))
     .when(pl.col("n_swaps") >= 20).then(pl.lit("active_retail"))
     .otherwise(pl.lit("casual_retail"))
     .alias("trader_type")
).sort("total_volume_usd", descending=True, nulls_last=True)

mev bots identified 9,097 representing 0.2281 percent
classification cutoffs whale size exceeds 24,937 and high frequency exceeds 73 swaps


---
## Dataset Export and Strict Quality Assertions
This cell writes the first version of the aggregated dataset to disk. It runs critical mathematical assertions to ensure that no trader profiles were duplicated during the joins and that all financial units represent logical fiat boundaries rather than raw cryptographic integers.

In [48]:
G_df.write_parquet(G_PATH, compression="zstd")
print(f"saved trader history base dataset size {G_PATH.stat().st_size/1e6:.1f} megabytes")

G_df.head(250_000).write_csv(OUT / "Dataset_G_TraderHistory_sample250k.csv")

print("trader type distribution overview")
print(G_df.group_by("trader_type").agg([
    pl.len().alias("n_traders"),
    (100 * pl.len() / G_df.height).alias("pct"),
    pl.col("total_volume_usd").sum().alias("volume_usd"),
    pl.col("median_trade_size_usd").median().alias("med_size"),
    pl.col("n_pools").median().alias("med_pools"),
    pl.col("hhi_volume").mean().alias("mean_hhi"),
    pl.col("victim_rate").mean().alias("mean_victim_rate"),
]).sort("n_traders", descending=True))

# strictly verify statistical and structural integrity
assert G_df["trader_address"].n_unique() == G_df.height, "duplicate traders detected"
assert G_df.select((pl.col("n_swaps") >= pl.col("n_transactions")).all()).item()
assert G_df.select((pl.col("n_pools") <= pl.col("n_swaps")).all()).item()
assert G_df.select((pl.col("tenure_days") >= 0).all()).item()
assert G_df.select((pl.col("victim_rate") <= 1.0).all()).item()

med = G_df["median_trade_size_usd"].median()
assert 1 < med < 1e6, f"median trade {med:,.0f} indicates usd units are severely miscalculated"
print(f"all structural assertions passed successfully median trade {med:,.2f}")

saved trader history base dataset size 518.2 megabytes
trader type distribution overview
shape: (6, 8)
┌────────────┬───────────┬───────────┬────────────┬────────────┬───────────┬──────────┬────────────┐
│ trader_typ ┆ n_traders ┆ pct       ┆ volume_usd ┆ med_size   ┆ med_pools ┆ mean_hhi ┆ mean_victi │
│ e          ┆ ---       ┆ ---       ┆ ---        ┆ ---        ┆ ---       ┆ ---      ┆ m_rate     │
│ ---        ┆ u32       ┆ f64       ┆ f64        ┆ f64        ┆ f64       ┆ f64      ┆ ---        │
│ str        ┆           ┆           ┆            ┆            ┆           ┆          ┆ f64        │
╞════════════╪═══════════╪═══════════╪════════════╪════════════╪═══════════╪══════════╪════════════╡
│ casual_ret ┆ 1978161   ┆ 49.606889 ┆ 1.0346e10  ┆ 112.244425 ┆ 2.0       ┆ NaN      ┆ 0.011094   │
│ ail        ┆           ┆           ┆            ┆            ┆           ┆          ┆            │
│ one_shot   ┆ 1802031   ┆ 45.190028 ┆ 2.3116e9   ┆ 50.0       ┆ 1.0       ┆ NaN      ┆ 0

---
## Analytical Sample Isolation and Anomaly Remediation
This cell prepares the dataset for formal regression analysis by isolating a pristine statistical sample. It mathematically neutralizes infinities and assigns a strict in_analysis_sample Boolean flag to profiles that meet high pricing coverage standards and exceed minimum dust value thresholds.

In [54]:
OUT = Path("./Thesis_Output")
G_PATH  = OUT / "Dataset_G_TraderHistory.parquet"
G2_PATH = OUT / "Dataset_G_TraderHistory_v2.parquet"

G_df = pl.read_parquet(G_PATH)

def finite(c):
    # converts non finite values like infinities into nulls to protect standard deviation math
    return pl.when(pl.col(c).is_finite()).then(pl.col(c)).otherwise(None).alias(c)

NUMERIC_FIX = ["hhi_volume", "top_pool_vol_share", "size_cv",
               "mev_loss_per_dollar", "log_total_volume", "log_avg_size",
               "trades_per_active_day", "activity_ratio"]

G_df = G_df.with_columns(
    pl.when(pl.col("usd_coverage") == 0).then(None)
      .otherwise(pl.col("total_volume_usd")).alias("total_volume_usd")
).with_columns([finite(c) for c in NUMERIC_FIX])

# define the core econometric analysis sample to avoid silent data drops
G_df = G_df.with_columns(
    (
        (pl.col("usd_coverage") >= 0.95)
        & pl.col("total_volume_usd").is_not_null()
        & (pl.col("total_volume_usd") > 0)
        & (pl.col("median_trade_size_usd") >= 1.0)      
    ).alias("in_analysis_sample")
)

print("analysis sample distribution")
print(G_df.group_by("in_analysis_sample").len())
print(G_df.filter("in_analysis_sample")
          .select(["hhi_volume", "top_pool_vol_share", "size_cv"]).describe())

analysis sample distribution
shape: (2, 2)
┌────────────────────┬─────────┐
│ in_analysis_sample ┆ len     │
│ ---                ┆ ---     │
│ bool               ┆ u32     │
╞════════════════════╪═════════╡
│ true               ┆ 3835938 │
│ false              ┆ 151736  │
└────────────────────┴─────────┘
shape: (9, 4)
┌────────────┬────────────┬────────────────────┬────────────┐
│ statistic  ┆ hhi_volume ┆ top_pool_vol_share ┆ size_cv    │
│ ---        ┆ ---        ┆ ---                ┆ ---        │
│ str        ┆ f64        ┆ f64                ┆ f64        │
╞════════════╪════════════╪════════════════════╪════════════╡
│ count      ┆ 3.835938e6 ┆ 3.835938e6         ┆ 2.135642e6 │
│ null_count ┆ 0.0        ┆ 0.0                ┆ 1.700296e6 │
│ mean       ┆ 0.775077   ┆ 0.81327            ┆ 0.709581   │
│ std        ┆ 0.281377   ┆ 0.244564           ┆ 0.576809   │
│ min        ┆ 0.000823   ┆ 0.005971           ┆ 0.0        │
│ 25%        ┆ 0.500001   ┆ 0.5707             ┆ 0.268149  

---
## Router Utilization and Concentrated Execution
This cell evaluates execution sophistication by analyzing the routers (e.g., Uniswap Universal Router, 1inch, MEV blocker contracts) utilized by each trader. It calculates a distinct router Herfindahl Hirschman Index, highlighting whether a user blindly submits to one interface or dynamically routes orders across the ecosystem.

In [57]:
A = pl.scan_parquet(OUT / "Dataset_A_Final.parquet").filter(
        ~pl.col("malformed_legs").fill_null(False))

# calculate the diversification of routing contracts used by the trader
router_tr = (
    A.select([pl.col("wallet").str.to_lowercase().alias("trader_address"),
              pl.col("sender").str.to_lowercase().alias("router")])
     .group_by(["trader_address", "router"]).agg(pl.len().alias("n"))
     .with_columns((pl.col("n") / pl.col("n").sum().over("trader_address")).alias("sh"))
     .group_by("trader_address").agg([
         pl.col("router").n_unique().alias("n_routers"),
         (pl.col("sh") ** 2).sum().alias("router_hhi"),
         pl.col("router").sort_by("n", descending=True).first().alias("top_router"),
         pl.col("sh").max().alias("top_router_share"),
     ])
).collect(engine="streaming")

G_df = G_df.join(router_tr, on="trader_address", how="left")

print("router volume leaderboard for manual verification")
print(A.group_by(pl.col("sender").str.to_lowercase().alias("router"))
       .agg([pl.len().alias("n_swaps"),
             pl.col("wallet").n_unique().alias("n_traders"),
             pl.col("amount_usd").sum().alias("vol_usd")])
       .sort("n_swaps", descending=True).head(30).collect(engine="streaming"))

router volume leaderboard for manual verification
shape: (30, 4)
┌─────────────────────────────────┬─────────┬───────────┬───────────┐
│ router                          ┆ n_swaps ┆ n_traders ┆ vol_usd   │
│ ---                             ┆ ---     ┆ ---       ┆ ---       │
│ str                             ┆ u32     ┆ u32       ┆ f64       │
╞═════════════════════════════════╪═════════╪═══════════╪═══════════╡
│ 0x66a9893cc07d91d95644aedd05d0… ┆ 4618416 ┆ 782532    ┆ 1.3940e10 │
│ 0xe592427a0aece92de3edee1f18e0… ┆ 3943619 ┆ 381241    ┆ 2.9218e10 │
│ 0xfbd4cdb413e45a52e2c8312f670e… ┆ 3761552 ┆ 24        ┆ 3.6930e10 │
│ 0x51c72848c68a965f66fa7a88855f… ┆ 3654941 ┆ 124       ┆ 6.4623e10 │
│ 0xa69babef1ca67a37ffaf7a485dff… ┆ 2027186 ┆ 499       ┆ 2.5065e10 │
│ …                               ┆ …       ┆ …         ┆ …         │
│ 0xd4bc53434c5e12cb41381a556c3c… ┆ 433519  ┆ 1         ┆ 5.9009e9  │
│ 0x5050e08626c499411b5d0e0b5af0… ┆ 419096  ┆ 49        ┆ 6.4287e9  │
│ 0x0d0e364aa7852291883c1

---
## Execution Quality and Network Cost Integration
This cell merges the precise pre trade price impacts derived in Dataset E and the exact network gas expenditures from Datasets B and C into the trader profile. This establishes a unified cost function for every trader, detailing their slippage efficiency against their priority fee generosity.

In [60]:
# integrate execution quality and slippage metrics from dataset e
E = (pl.scan_parquet(OUT / "Dataset_E_PoolState.parquet")
       .select([pl.col("transaction_hash").str.to_lowercase().alias("tx_hash"),
                "log_index", "price_impact"]))

exec_tr = (
    A.select([pl.col("wallet").str.to_lowercase().alias("trader_address"),
              pl.col("tx_hash").str.to_lowercase(), "log_index"])
     .join(E, on=["tx_hash", "log_index"], how="inner")
     .with_columns(pl.col("price_impact").abs().alias("abs_pi"))
     .group_by("trader_address").agg([
         pl.col("abs_pi").mean().alias("mean_price_impact"),
         pl.col("abs_pi").median().alias("med_price_impact"),
         pl.col("abs_pi").quantile(0.90).alias("p90_price_impact"),
         pl.len().alias("n_swaps_with_pi"),
     ])
).collect(engine="streaming")

# integrate transaction execution costs leveraging network base fees and effective prices
B = pl.scan_parquet(OUT / "Dataset_B_Final.parquet")
C = pl.scan_parquet(OUT / "Dataset_C_Final.parquet").select(["block_number", "base_fee"])

tx_px = (A.group_by(pl.col("tx_hash").str.to_lowercase())
           .agg([pl.col("wallet").str.to_lowercase().first().alias("trader_address"),
                 pl.col("weth_price_at_block").first().alias("weth_px"),
                 pl.col("amount_usd").sum().alias("tx_vol_usd")]))

gas_tr = (
    B.join(C, on="block_number", how="left")
     .with_columns([
         (pl.col("fee_wei").cast(pl.Float64) / 1e18).alias("fee_eth"),
         (pl.max_horizontal(
              pl.col("effective_gas_price").cast(pl.Float64) - pl.col("base_fee").cast(pl.Float64),
              pl.lit(0.0)) / 1e9).alias("priority_gwei"),
     ])
     .join(tx_px, on="tx_hash", how="inner")
     .with_columns((pl.col("fee_eth") * pl.col("weth_px")).alias("gas_usd"))
     .group_by("trader_address").agg([
         pl.col("gas_usd").sum().alias("total_gas_usd"),
         pl.col("gas_usd").median().alias("med_gas_usd"),
         pl.col("priority_gwei").median().alias("med_priority_gwei"),
         (pl.col("gas_usd").sum() / pl.col("tx_vol_usd").sum()).alias("gas_cost_ratio"),
     ])
).collect(engine="streaming")

G_df = (G_df.join(exec_tr, on="trader_address", how="left")
            .join(gas_tr,  on="trader_address", how="left"))

G_df.write_parquet(G2_PATH, compression="zstd")
print(f"version two saved size {G2_PATH.stat().st_size/1e6:.1f} megabytes")

version two saved size 720.2 megabytes


---
## Transaction Level Mev Reconciliation
This cell performs a rigorous reconciliation pass. Instead of analyzing Mev at the individual log level, it elevates the analysis to the parent transaction level, recalculating extraction rates across the entire transaction hash. This eliminates false assumptions caused by multileg token hops and significantly increases the accuracy of victim identification.

In [61]:
OUT = Path("./Thesis_Output")
G2_PATH = OUT / "Dataset_G_TraderHistory_v2.parquet"
G3_PATH = OUT / "Dataset_G_TraderHistory_v3.parquet"

G_df = pl.read_parquet(G2_PATH)

# aggregate mev events up to the parent transaction boundary
F_tx = (
    pl.scan_parquet(OUT / "Dataset_F_MEV.parquet")
      .with_columns(pl.col("transaction_hash").str.to_lowercase().alias("tx_hash"))
      .group_by("tx_hash").agg([
          pl.col("is_sandwich_victim").fill_null(False).any().alias("v"),
          pl.col("is_sandwich_attacker").fill_null(False).any().alias("a"),
          pl.col("is_arbitrage").fill_null(False).any().alias("arb"),
          pl.col("estimated_loss_usd").sum().alias("loss_usd"),
      ])
)

A_tx = (
    pl.scan_parquet(OUT / "Dataset_A_Final.parquet")
      .filter(~pl.col("malformed_legs").fill_null(False))
      .group_by(pl.col("tx_hash").str.to_lowercase())
      .agg(pl.col("wallet").str.to_lowercase().first().alias("trader_address"))
)

cov = (F_tx.join(A_tx, on="tx_hash", how="left")
           .select([pl.len().alias("f_tx"),
                    pl.col("trader_address").is_null().sum().alias("unmatched")])
           .collect(engine="streaming"))
print(f"transaction level unmatched {100*cov['unmatched'][0]/cov['f_tx'][0]:.2f} percent")

mev_tx = (
    A_tx.join(F_tx, on="tx_hash", how="inner")
        .group_by("trader_address").agg([
            pl.col("v").sum().alias("n_victim_tx"),
            pl.col("a").sum().alias("n_attacker_tx"),
            pl.col("arb").sum().alias("n_arb_tx"),
            pl.col("loss_usd").sum().alias("mev_loss_usd_v2"),
        ])
).collect(engine="streaming")

# replace legacy log metrics with accurate parent transaction metrics
G_df = (
    G_df.drop([c for c in ["n_victim_tx","n_attacker_tx","n_arb_tx","mev_loss_usd_v2"]
               if c in G_df.columns])
        .join(mev_tx, on="trader_address", how="left")
        .with_columns([pl.col(c).fill_null(0) for c in
                       ["n_victim_tx","n_attacker_tx","n_arb_tx"]])
        .with_columns([
            pl.col("mev_loss_usd_v2").fill_null(0.0),
            (pl.col("n_victim_tx")   / pl.col("n_transactions")).alias("victim_rate_tx"),
            (pl.col("n_attacker_tx") / pl.col("n_transactions")).alias("attacker_rate_tx"),
            (pl.col("n_arb_tx")      / pl.col("n_transactions")).alias("arb_rate_tx"),
        ])
)

print(G_df.filter("in_analysis_sample").group_by("trader_type").agg([
    pl.len().alias("n"),
    pl.col("victim_rate").mean().alias("victim_rate_OLD_logidx"),
    pl.col("victim_rate_tx").mean().alias("victim_rate_NEW_tx"),
    pl.col("cyclic_share").mean().alias("cyclic_share"),
    pl.col("arb_rate_tx").mean().alias("arb_rate_F"),
]).sort("n", descending=True))

transaction level unmatched 2.44 percent
shape: (6, 6)
┌────────────────┬─────────┬──────────────────────┬────────────────────┬──────────────┬────────────┐
│ trader_type    ┆ n       ┆ victim_rate_OLD_logi ┆ victim_rate_NEW_tx ┆ cyclic_share ┆ arb_rate_F │
│ ---            ┆ ---     ┆ dx                   ┆ ---                ┆ ---          ┆ ---        │
│ str            ┆ u32     ┆ ---                  ┆ f64                ┆ f64          ┆ f64        │
│                ┆         ┆ f64                  ┆                    ┆              ┆            │
╞════════════════╪═════════╪══════════════════════╪════════════════════╪══════════════╪════════════╡
│ casual_retail  ┆ 1930743 ┆ 0.011309             ┆ 0.015676           ┆ 0.000969     ┆ 0.299944   │
│ one_shot       ┆ 1700296 ┆ 0.008724             ┆ 0.008854           ┆ 0.0          ┆ 0.024101   │
│ active_retail  ┆ 134424  ┆ 0.014144             ┆ 0.01992            ┆ 0.001544     ┆ 0.286187   │
│ high_frequency ┆ 34758   ┆ 0.01401

---
## Size Coefficient of Variation Refinement
This cell executes a quick cleanup of the size coefficient of variation. This metric measures how wildly a trader’s trade size fluctuates. By mathematically requiring at least three priced swaps to define the variance, it prevents statistical anomalies from polluting the behavior profiles of one shot retail users.

In [65]:
G_df = G_df.with_columns([
    # Establish a boolean flag confirming the trader has a statistically valid sample size
    (pl.col("n_swaps_priced") >= 3).alias("size_cv_defined"),
    
    # Require a minimum of three priced swaps to mathematically justify a variance calculation
    pl.when(pl.col("n_swaps_priced") >= 3).then(pl.col("size_cv"))
      .otherwise(None).alias("size_cv_clean"),
])

# Print the sanitized descriptive statistics for the core analysis sample
print(G_df.filter(pl.col("in_analysis_sample") & pl.col("size_cv_defined"))
          .select("size_cv_clean").describe())

shape: (9, 2)
┌────────────┬───────────────┐
│ statistic  ┆ size_cv_clean │
│ ---        ┆ ---           │
│ str        ┆ f64           │
╞════════════╪═══════════════╡
│ count      ┆ 1.377093e6    │
│ null_count ┆ 0.0           │
│ mean       ┆ 0.862022      │
│ std        ┆ 0.568868      │
│ min        ┆ 0.0           │
│ 25%        ┆ 0.517159      │
│ 50%        ┆ 0.795849      │
│ 75%        ┆ 1.121024      │
│ max        ┆ 39.092893     │
└────────────┴───────────────┘


---
## Workspace Finalization and Legacy Cleanup
This final cell performs essential repository housekeeping. Throughout the pipeline, we generated several intermediate diagnostic files and versioned datasets (such as version one and version two of the trader history). This script safely deletes those deprecated artifacts to free up hard drive space and prevent accidental usage of outdated data. Finally, it renames the fully enriched version two dataset into the canonical Dataset_G_TraderHistory.parquet and generates a massive, universal CSV file for seamless compatibility with external statistical software like Stata or R.

In [68]:
OUT = Path("./Thesis_Output")

# Identify the exact file paths for the finalization routine
true_g_v2 = OUT / "Dataset_G_TraderHistory_v2.parquet"
final_g_parquet = OUT / "Dataset_G_TraderHistory.parquet"
final_g_csv = OUT / "Dataset_G_TraderHistory.csv"

# Catalog intermediate and diagnostic files that are no longer required
junk_files = [
    OUT / "Dataset_G_TraderHistory_sample250k.csv",
    OUT / "router_leaderboard.csv",
    OUT / "Dataset_H_SwapPanel.parquet"  
]

print("Starting analytical workspace cleanup")

# Delete the deprecated version one file to prevent naming collisions
if final_g_parquet.exists() and true_g_v2.exists():
    os.remove(final_g_parquet)
    print(f"Deleted legacy version one file {final_g_parquet.name}")

# Promote the fully enriched version two file to the canonical dataset name
if true_g_v2.exists():
    os.rename(true_g_v2, final_g_parquet)
    print(f"Renamed {true_g_v2.name} to {final_g_parquet.name}")
else:
    print(f"Could not find {true_g_v2.name} as it may have already been renamed")

# Iterate through the junk catalog and delete unnecessary intermediate artifacts
for f in junk_files:
    if f.exists():
        os.remove(f)
        print(f"Deleted deprecated file {f.name}")
    else:
        print(f"File already removed skipping {f.name}")

# Convert the finalized parquet dataset into a universal csv format
print(f"Converting {final_g_parquet.name} to CSV format")
if final_g_parquet.exists():
    
    # Utilizing Polars to handle the massive memory footprint of a four million row dataframe
    df = pl.read_parquet(final_g_parquet)
    df.write_csv(final_g_csv)
    
    print(f"CSV successfully created {final_g_csv.name}")
    print(f"Final file size {final_g_csv.stat().st_size / (1024**3):.2f} gigabytes")
else:
    print("Error Final Parquet file not found to convert")
    
print("Directory is clean and the final trader demographic dataset is ready for analysis")

Starting analytical workspace cleanup
Deleted legacy version one file Dataset_G_TraderHistory.parquet
Renamed Dataset_G_TraderHistory_v2.parquet to Dataset_G_TraderHistory.parquet
Deleted deprecated file Dataset_G_TraderHistory_sample250k.csv
File already removed skipping router_leaderboard.csv
File already removed skipping Dataset_H_SwapPanel.parquet
Converting Dataset_G_TraderHistory.parquet to CSV format
CSV successfully created Dataset_G_TraderHistory.csv
Final file size 2.53 gigabytes
Directory is clean and the final trader demographic dataset is ready for analysis


---
## Results and Data Integrity

I successfully synthesized over sixty million swaps into a finalized demographic panel representing millions of unique trading entities. implemented strict data sanitization passes to eliminate mathematical infinities and dust values, generating a Boolean flag to explicitly define the core analysis sample. 